# Visual servo — capture + flight loop

`servo.py` is camera-free: it never opens a `VideoCapture`, reads a frame or shows a
window. **This notebook owns the camera and the loop** and feeds frames into it.

Frame-rate reality check for this rig: Logitech webcams expose only `YUYV` and `MJPG`
over UVC — there is no mono/`GREY` format to stream — and the consumer models
(C270/C920/C930e) are capped at **30 fps at every resolution**, so lowering the
resolution buys no rate. Only the C922 (720p60) and Brio (1080p60) reach 60.

What *does* help, and is done below:

- **`MJPG` fourcc.** Without it V4L2 negotiates `YUYV`; 1280x720 `YUYV` at 30 fps needs
  ~442 Mbit/s, past the ~320 Mbit/s a USB 2.0 isochronous pipe delivers in practice, so
  the camera drops to a rate that fits. (macOS AVFoundation ignores the fourcc and picks
  MJPG on its own — this matters on Linux.)
- **Grayscale once on the host** (`GRAY = True`) so MOG2 and both morphology passes
  run on 1 channel instead of 3. It does not lift the 30 fps ceiling, but it keeps
  the processing loop from becoming the bottleneck.
- **`CAP_PROP_BUFFERSIZE = 1`**, so latency does not pile up in the driver queue.

**Measured on the C270 here:** ~28 fps at *every* resolution from 160x120 to 1280x720 —
flat within 3%. **Resolution is free, so run the full 1280x720**; lowering it costs
`px_per_mm` precision and buys no rate. Requesting 60 or 120 clamps to 30, while
requesting 15 does give 15 — so the cap is the hardware, not an ignored property.


In [ ]:
import glob
import os
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.abspath(""))
import servo

In [ ]:
# Every USB camera registers TWO /dev/video* nodes: a capture node and a metadata
# node that cannot stream (sysfs "index" 0 and 1). Minor numbers are handed out in
# plug order and shift whenever you replug -- the C270 was video4, the Dino-Lite
# came up as video5. Match the device NAME, never a hardcoded number.
CAM_NAME = "c270"           # substring, case-insensitive; None = first camera found
FRAME_W, FRAME_H, FPS = 1280, 720, 30
GRAY = True                 # 1-channel MOG2 + morphology instead of 3


def list_cameras():
    """[(index, name)] for every V4L2 capture node (metadata nodes excluded)."""
    found = []
    for path in sorted(glob.glob("/sys/class/video4linux/video*")):
        try:
            with open(os.path.join(path, "index")) as f:
                if int(f.read()) != 0:          # 1 = metadata node, can't stream
                    continue
            with open(os.path.join(path, "name")) as f:
                found.append((int(os.path.basename(path)[5:]), f.read().strip()))
        except OSError:
            continue
    return found


_last_cap = []          # release stale handles: a cell interrupted before
                        # cap.release() leaves a VideoCapture bound to a node
                        # that may vanish on replug ("failed VIDIOC_REQBUFS:
                        # errno=19 No such device" from the old object).

# Free function in OpenCV 4.x, a static method in 5.x.
_fourcc = getattr(cv2, "VideoWriter_fourcc", None) or cv2.VideoWriter.fourcc


def fourcc_str(cap):
    v = int(cap.get(cv2.CAP_PROP_FOURCC))
    return "".join(chr((v >> 8 * i) & 0xFF) for i in range(4))


def open_camera(name=None, index=None, w=FRAME_W, h=FRAME_H, fps=FPS, mjpg=True):
    """Open by name substring (default CAM_NAME) or explicit index.

    Prints the mode actually negotiated: cap.set() returning True does NOT mean
    the camera granted it, and a silent fallback to YUYV is exactly the failure
    that costs you the frame rate.
    """
    while _last_cap:
        _last_cap.pop().release()
    cameras = list_cameras()
    if index is None:
        want = CAM_NAME if name is None else name
        matches = [i for i, n in cameras if want is None or want.lower() in n.lower()]
        if not matches:
            raise RuntimeError(f"no camera matching {want!r}; found {cameras}")
        index = matches[0]

    cap = cv2.VideoCapture(index, cv2.CAP_V4L2)   # pinned: no silent FFMPEG fallback
    # FOURCC FIRST -- it has to be set before the frame size, or the driver
    # re-negotiates the format and drops back to YUYV.
    if mjpg:
        cap.set(cv2.CAP_PROP_FOURCC, _fourcc(*"MJPG"))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
    cap.set(cv2.CAP_PROP_FPS, fps)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    if not (cap.isOpened() and cap.read()[0]):
        cap.release()
        raise RuntimeError(
            f"/dev/video{index} won't stream (already open in another cell?); "
            f"capture nodes: {cameras}"
        )
    _last_cap.append(cap)
    print(f"opened /dev/video{index}: {dict(cameras).get(index, '?')}")
    print(
        f"  negotiated {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}"
        f"x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))} {fourcc_str(cap)}"
        f" @ {cap.get(cv2.CAP_PROP_FPS):g} fps"
    )
    return cap


def grab(cap):
    """One frame, gray if GRAY. None if the read failed."""
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if GRAY and frame.ndim == 3 else frame


def measure_fps(cap, n=120):
    """Time n real reads. cap.get(CAP_PROP_FPS) is only what the driver CLAIMS."""
    grab(cap)                                   # discard: first read pays startup
    t0 = time.monotonic()
    got = sum(grab(cap) is not None for _ in range(n))
    dt = time.monotonic() - t0
    print(
        f"{got}/{n} frames in {dt:.2f} s -> {got / dt:.1f} fps measured "
        f"(driver claims {cap.get(cv2.CAP_PROP_FPS):g})"
    )
    return got / dt


list_cameras()

## 1. Open the camera and measure what you actually get

Run this, then re-run with `open_camera(mjpg=False)` to see what the `MJPG` fourcc is
worth on this camera. `measure_fps` is the only honest number — the driver's claimed
FPS is frequently not what the loop sees.

In [ ]:
cap = open_camera()
measure_fps(cap)

## 2. Serial link

`connect()` pulses EN so the board starts from a known state, then waits for boot.

In [ ]:
link = servo.connect()

## 3. Calibrate scale and datum

**Coils off.** Move the robot around by hand for the whole pass: MOG2 only sees moving
foreground, and this same pass is what settles the background model.

`px_per_mm` comes from the robot's projected diameter, so it measures *distance to
camera*, not height — it only holds while the robot stays in that plane. If the median
`axis_px` warning fires, move the camera closer rather than raising the resolution.

In [ ]:
N_CAL = 300

bg = servo.new_bg()
dets = []
for _ in range(N_CAL):
    frame = grab(cap)
    if frame is None:
        continue
    d = servo.detect(bg, frame)
    dets.append(d)                       # Nones are fine, scale_from_detections drops them
    cv2.imshow("calibrate", servo.annotate(frame, d))
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cv2.destroyWindow("calibrate")

px_per_mm, datum_v, axes = servo.scale_from_detections(dets)
print(f"px_per_mm = {px_per_mm:.3f}   datum_v = {datum_v:.1f} px")

## 4. Altitude hold

**The whole loop must stay in this one cell.** The firmware has no watchdog, so
`servo.coils_on(link)` is the only thing that brings the coils down when the loop
exits — including on a kernel interrupt. Split the loop across cells and interrupting
leaves the coils energised with nothing driving them.

Start with a short `DURATION_S`. Takeoff needs no spin-up ramp: the incremental PID
accumulates the steady error on the pad and walks the frequency up until the robot
lifts.

In [ ]:
Z_REF_MM = 30.0
DURATION_S = 5.0                          # keep the first run short

ctrl = servo.height_controller()          # kp=0.2, ki=0.15, kd=0.01
step = servo.altitude_hold(link, bg, px_per_mm, datum_v, ctrl, z_ref_mm=Z_REF_MM)

rows = []
t0 = time.monotonic()
with servo.coils_on(link):                # coils come down however we leave
    while time.monotonic() - t0 < DURATION_S:
        frame = grab(cap)
        if frame is None:
            continue
        row, d = step(frame)              # pass z_ref=... here for a step response
        rows.append(row)
        cv2.imshow("servo", servo.annotate(frame, d, row["z_mm"], row["zdot"]))
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
cv2.destroyWindow("servo")

held = sum(r["z_mm"] is not None for r in rows)
print(
    f"{len(rows)} samples over {time.monotonic() - t0:.1f} s "
    f"({len(rows) / (time.monotonic() - t0):.1f} fps, {held} with a lock); coils off"
)

## 5. Plot

In [ ]:
def col(key):
    """Column as float, with None -> nan so gaps break the line instead of raising."""
    return np.array([np.nan if r[key] is None else r[key] for r in rows], float)


t = col("t")
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 6))
ax1.plot(t, col("z_mm"), ".-", ms=3, lw=0.8, label="z")
ax1.plot(t, col("z_ref"), "--", label="z_ref")
ax1.set_ylabel("height (mm)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, col("u_hz"), lw=0.8, label="commanded")
ax2.plot(t, col("freq_meas"), ".", ms=3, label="telemetry")
ax2.set_ylabel("rotation freq (Hz)")
ax2.set_xlabel("t (s)")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()

## 6. Release

Run this when you are done, and after any interrupted cell.

In [ ]:
servo.stop(link)
cv2.destroyAllWindows()
while _last_cap:
    _last_cap.pop().release()
print("coils off, windows closed, camera released")